# ULAL - full pipeline (30 classes, best methods)

The whole project in one run, on a 30-class stream, using only the winning method at
each stage:

```
ResNet50 + LSTM      trained on the 10 base classes
   -> Replay + LwF   continual learning to 30 classes (base -> +10 -> +10)
   -> CE + KD        distilled into a MobileNet student
   -> Reptile        adapted to real YouTube clips
```

The class configuration is switched to 30 in the Setup cell below - the study notebooks
use 20 to stay fast; this capstone uses the full stream.

## **Connect Colab**

In [ ]:
import os, sys, subprocess
from getpass import getpass

REPO   = "silvergjeka22/Unified-Lifelong-Action-Learning"
BRANCH = "real-video"
ROOT   = "/content/ulal"

token = os.environ.get("GITHUB_TOKEN") or getpass("GitHub token: ")
os.environ["GITHUB_TOKEN"] = token

if os.path.isdir(os.path.join(ROOT, ".git")):
    subprocess.run(["git", "-C", ROOT, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", ROOT, "reset", "--hard", "FETCH_HEAD"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "--branch", BRANCH,
                    "https://" + token + "@github.com/" + REPO + ".git", ROOT], check=True)

if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

In [ ]:
os.environ["KAGGLE_USERNAME"] = "colab"
os.environ["KAGGLE_KEY"]      = "KEY"

## **Download DATASET**

In [ ]:
from src.bootstrap import setup
setup(drive=True, dataset=True)

In [ ]:
!pip install -q yt-dlp

In [ ]:
# imports
%run /content/ulal/src/imports.py
set_seed(cfg.SEED)

## **Setup (30 classes)**

Switch the class configuration to the full 10 + 10 + 10 stream, in the notebook, before anything is built.

In [ ]:
cfg.TASK_1 = ["ApplyEyeMakeup", "ApplyLipstick", "Archery", "BabyCrawling", "BalanceBeam",
              "BandMarching", "BlowDryHair", "BlowingCandles", "BodyWeightSquats", "Bowling"]
cfg.TASK_2 = ["BoxingPunchingBag", "BoxingSpeedBag", "BrushingTeeth", "CliffDiving", "CricketBowling",
              "CricketShot", "CuttingInKitchen", "FieldHockeyPenalty", "Haircut", "SoccerPenalty"]

In [ ]:
ALL_CLASSES_LIST = cfg.SELECTED_CLASSES + cfg.TASK_1 + cfg.TASK_2
class_to_idx     = {cls_name: i for i, cls_name in enumerate(ALL_CLASSES_LIST)}
n_base = len(cfg.SELECTED_CLASSES)
n_t1   = len(cfg.TASK_1)
n_t2   = len(cfg.TASK_2)
print(f"base {n_base}  task1 {n_t1}  task2 {n_t2}  total {len(ALL_CLASSES_LIST)}")

## **Preprocess data**

In [ ]:
base_split  = build_group_split(cfg.SELECTED_CLASSES, train_groups=18, val_groups=3)
task1_split = build_group_split(cfg.TASK_1,           train_groups=18, val_groups=3)
task2_split = build_group_split(cfg.TASK_2,           train_groups=18, val_groups=3)

In [ ]:
# BASE  (max_samples caps TRAIN clips/class via cfg.MAX_SAMPLES; None = full data, val/test untouched)
preprocess_group_split(base_split, cfg.SELECTED_CLASSES, output_root=cfg.BASE_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# TASK 1  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task1_split, cfg.TASK_1, output_root=cfg.TASK1_ROOT, max_samples=cfg.MAX_SAMPLES)

In [ ]:
# TASK 2  (same TRAIN-only cap via cfg.MAX_SAMPLES)
preprocess_group_split(task2_split, cfg.TASK_2, output_root=cfg.TASK2_ROOT, max_samples=cfg.MAX_SAMPLES)

## **Create Loaders**

In [ ]:
base_train  = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "train"), class_to_idx=class_to_idx)
base_val    = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "val"),   class_to_idx=class_to_idx)
base_test   = UCF101Clips(os.path.join(cfg.BASE_ROOT,  "test"),  class_to_idx=class_to_idx)
task1_train = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "train"), class_to_idx=class_to_idx)
task1_val   = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "val"),   class_to_idx=class_to_idx)
task1_test  = UCF101Clips(os.path.join(cfg.TASK1_ROOT, "test"),  class_to_idx=class_to_idx)
task2_train = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "train"), class_to_idx=class_to_idx)
task2_val   = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "val"),   class_to_idx=class_to_idx)
task2_test  = UCF101Clips(os.path.join(cfg.TASK2_ROOT, "test"),  class_to_idx=class_to_idx)

base_train_loader = DataLoader(base_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
base_val_loader   = DataLoader(base_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
base_test_loader  = DataLoader(base_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_train_loader = DataLoader(task1_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
task1_val_loader   = DataLoader(task1_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task1_test_loader  = DataLoader(task1_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_train_loader = DataLoader(task2_train, batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
task2_val_loader   = DataLoader(task2_val,   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
task2_test_loader  = DataLoader(task2_test,  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

combined_test_loader  = DataLoader(ConcatDataset([base_test, task1_test]),              batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)
combined_test_loader2 = DataLoader(ConcatDataset([base_test, task1_test, task2_test]),  batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

print(f"base {len(base_train)}/{len(base_val)}/{len(base_test)}  "
      f"task1 {len(task1_train)}/{len(task1_val)}/{len(task1_test)}  "
      f"task2 {len(task2_train)}/{len(task2_val)}/{len(task2_test)}")

## **Backbone: ResNet50 on base**

Train the ResNet50 + LSTM on the 10 base classes - the teacher every later stage builds on.

In [ ]:
teacher = ResNet50LSTM(num_classes=n_base).to(device)
set_seed(cfg.SEED)
hist_base = train_model(teacher, base_train_loader, base_val_loader, num_epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR, device=device)

In [ ]:
plot_training_results(hist_base, task_name="ResNet50 base")

In [ ]:
base_acc, base_preds, base_labels = test_model(teacher, base_test_loader, device)
plot_confusion_matrix(base_labels, base_preds, num_classes=n_base,
                      classes_list=cfg.SELECTED_CLASSES, name="ResNet50 base")

## **Continual Learning: Replay + LwF**

Grow the head base -> 20 -> 30, protected by both a replay buffer and a frozen teacher (the strongest arm from section 3).

In [ ]:
model = unfreeze_all(teacher)
snapshots = []
forget_hist = []

In [ ]:
buffer = ReplayBuffer(max_size=600)
fill_buffer(buffer, base_train_loader, per_class=15)
teacher_t1 = snapshot_teacher(model)
print(f"buffer: {len(buffer.data)} clips")

In [ ]:
model = expand_classifier(model, n_base + n_t1)
model = unfreeze_all(model).to(device)
snapshots.append(evaluate_all_tasks(model, device, base_test_loader, [task1_test_loader], [combined_test_loader]))

In [ ]:
set_seed(cfg.SEED)
track1 = {"Base": base_test_loader, "Task 1": task1_test_loader}
train_t1 = train_continual(model, task1_train_loader, task1_val_loader, device,
                           buffer=buffer, teacher=teacher_t1, num_old_classes=n_base,
                           lambda_distill=cfg.LAMBDA_DISTILL, T=cfg.KD_TEMPERATURE,
                           epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR,
                           track_loaders=track1, track_history=forget_hist)

In [ ]:
plot_training_results(train_t1, task_name="Task 1 - Replay + LwF")
snapshots.append(evaluate_all_tasks(model, device, base_test_loader, [task1_test_loader], [combined_test_loader]))

In [ ]:
teacher_t2 = snapshot_teacher(model)
fill_buffer(buffer, task1_train_loader, per_class=15)
model = expand_classifier(model, n_base + n_t1 + n_t2)
model = unfreeze_all(model).to(device)
print(f"buffer: {len(buffer.data)} clips  |  head out={model.fc.out_features}")

In [ ]:
set_seed(cfg.SEED)
track2 = {"Base": base_test_loader, "Task 1": task1_test_loader, "Task 2": task2_test_loader}
train_t2 = train_continual(model, task2_train_loader, task2_val_loader, device,
                           buffer=buffer, teacher=teacher_t2, num_old_classes=n_base + n_t1,
                           lambda_distill=cfg.LAMBDA_DISTILL, T=cfg.KD_TEMPERATURE,
                           epochs=cfg.CL_EPOCHS, lr=cfg.CL_LR,
                           track_loaders=track2, track_history=forget_hist)

In [ ]:
plot_training_results(train_t2, task_name="Task 2 - Replay + LwF")
snapshots.append(evaluate_all_tasks(model, device, base_test_loader,
                                    [task1_test_loader, task2_test_loader],
                                    [combined_test_loader, combined_test_loader2]))

In [ ]:
cl_acc, cl_preds, cl_labels = test_model(model, combined_test_loader2, device)
plot_confusion_matrix(cl_labels, cl_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="Continual (30 classes)")

In [ ]:
plot_forgetting_history(forget_hist, epochs_per_task=cfg.CL_EPOCHS)

## **Distillation: CE + KD**

Distil the 30-class continual model into a MobileNet student, trained on all 30 classes at once.

In [ ]:
train_all_loader = DataLoader(ConcatDataset([base_train, task1_train, task2_train]), batch_size=cfg.BATCH_SIZE, shuffle=True,  num_workers=2)
val_all_loader   = DataLoader(ConcatDataset([base_val,   task1_val,   task2_val]),   batch_size=cfg.BATCH_SIZE, shuffle=False, num_workers=2)

In [ ]:
student = MobileNetV3SmallLSTMStudent(num_classes=len(ALL_CLASSES_LIST)).to(device)
set_seed(cfg.SEED)
student, hist_kd = train_student(student, model, train_all_loader, val_all_loader, device,
                                 mode="kd", T=cfg.KD_T, ce_weight=cfg.KD_CE_WEIGHT,
                                 distill_weight=cfg.KD_DISTILL_WEIGHT,
                                 epochs=cfg.KD_EPOCHS, lr=cfg.KD_LR)

In [ ]:
plot_training_results(hist_kd, task_name="student CE + KD")

In [ ]:
student_acc, student_preds, student_labels = test_model(student, combined_test_loader2, device)
plot_confusion_matrix(student_labels, student_preds, num_classes=len(ALL_CLASSES_LIST),
                      classes_list=ALL_CLASSES_LIST, name="Student CE + KD")

In [ ]:
teacher_params = sum(p.numel() for p in model.parameters())
student_params = sum(p.numel() for p in student.parameters())
print(f"continual model : {cl_acc:.4f}  ({teacher_params/1e6:.1f}M)")
print(f"student CE+KD    : {student_acc:.4f}  ({student_params/1e6:.1f}M, {teacher_params/student_params:.0f}x smaller)")

## **Domain Adaptation (few-shot)**

Adapt the deployed student to real YouTube clips from a few labelled clips per class. This is the same few-shot adaptation MAML meta-tests with; the full MAML meta-learning study is in `study/5_domain_adapt/meta_domain_adapt.ipynb`.

In [ ]:
yt_clips, yt_labels, _ = download_youtube_clips(cfg.YOUTUBE_CLIPS, class_to_idx)
print(f"youtube clips: {tuple(yt_clips.shape)}  classes: {sorted(set(yt_labels.tolist()))}")

In [ ]:
yt_ids = sorted(set(yt_labels.tolist()))
sx_list, sy_list, qx_list, qy_list = [], [], [], []
for c in yt_ids:
    idx  = (yt_labels == c).nonzero(as_tuple=True)[0]
    perm = idx[torch.randperm(len(idx))]
    sx_list.append(yt_clips[perm[:5]]); sy_list = sy_list + [c] * 5
    qx_list.append(yt_clips[perm[5:]]); qy_list = qy_list + [c] * (len(perm) - 5)

yt_sx = torch.cat(sx_list).to(device)
yt_sy = torch.tensor(sy_list).to(device)
yt_qx = torch.cat(qx_list)
yt_qy = torch.tensor(qy_list)
yt_test_loader = DataLoader(TensorDataset(yt_qx, yt_qy), batch_size=cfg.BATCH_SIZE, shuffle=False)
print(f"support {tuple(yt_sx.shape)}  query {tuple(yt_qx.shape)}")

In [ ]:
yt_before, _, _    = test_model(student, yt_test_loader, device)
clean_before, _, _ = test_model(student, base_test_loader, device)
print(f"before | youtube: {yt_before:.4f} | clean UCF: {clean_before:.4f}")

In [ ]:
set_seed(cfg.SEED)
adapted_student = few_shot_adapt(student, yt_sx, yt_sy, 0.01, 10)

In [ ]:
yt_after, _, _    = test_model(adapted_student, yt_test_loader, device)
clean_after, _, _ = test_model(adapted_student, base_test_loader, device)
print(f"after  | youtube: {yt_after:.4f} (gain {yt_after - yt_before:+.4f}) | clean UCF: {clean_after:.4f} (change {clean_after - clean_before:+.4f})")

## **Summary**

In [ ]:
print("ULAL pipeline - 30 classes")
print(f"  ResNet50 base (10)        : {base_acc:.4f}")
print(f"  Replay + LwF (30, all)    : {cl_acc:.4f}")
print(f"  MobileNet student (CE+KD) : {student_acc:.4f}  ({teacher_params/student_params:.0f}x smaller)")
print(f"  YouTube before -> after   : {yt_before:.4f} -> {yt_after:.4f}")
print(f"  clean UCF kept            : {clean_after:.4f}")

In [ ]:
os.makedirs(cfg.RESULTS_DIR, exist_ok=True)
payload = {"base_acc": base_acc, "continual_acc": cl_acc, "student_acc": student_acc,
           "youtube_before": yt_before, "youtube_after": yt_after,
           "clean_before": clean_before, "clean_after": clean_after,
           "teacher_params": teacher_params, "student_params": student_params}
with open(f"{cfg.RESULTS_DIR}/ulal_summary.json", "w") as f:
    json.dump(payload, f, indent=2)
print(f"saved -> {cfg.RESULTS_DIR}/ulal_summary.json")